# Interpretación del Modelo e Impacto de Negocio
En este notebook nos enfocaremos en traducir el modelo matemático a términos de negocio. Realizaremos dos tareas principales:
1. **Interpretación del Modelo (Feature Importance):** Entenderemos cuáles son las variables que más influyen en la decisión de cancelar una reserva.
2. **Impacto Económico:** Simularemos cómo el uso de nuestro modelo (con el umbral ajustado) impacta financieramente al hotel comparado con no hacer nada.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

import warnings
warnings.filterwarnings('ignore')

### 1. Carga de Datos y Entrenamiento del Modelo Definitivo
Entrenaremos nuestro Random Forest ganador utilizando el 100% de los datos históricos (`X_train_full`).

In [ ]:
base_path = '../data/model_input/cv/'

# Cargamos TODO el conjunto de entrenamiento histórico
X_train_full = pd.read_csv(base_path + 'X_train_full.csv')
y_train_full = pd.read_csv(base_path + 'y_train_full.csv').values.ravel()

print(f"Entrenando con {X_train_full.shape[0]} reservas históricas...")

In [ ]:
# 1. Definir el Preprocesador
vars_categoricas = ['meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type']
vars_numericas = ['lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 'previous_cancellations', 'required_car_parking_spaces', 'total_of_special_requests', 'agent', 'company', 'arrival_date_year', 'arrival_month_num']
vars_binarias = ['is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)

# 2. Definir los hiperparámetros ganadores 
rf_best_params = {
    'n_estimators': 100,
    'max_depth': 30,
    'min_samples_split': 2,
    'min_samples_leaf': 4,
    'random_state': 42,
    'n_jobs': -1
}

# 3. Crear y entrenar el Pipeline Definitivo
rf_pipeline_final = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(**rf_best_params))
])

rf_pipeline_final.fit(X_train_full, y_train_full)
print("¡Modelo definitivo entrenado exitosamente!")

### 2. Interpretación del Modelo: Feature Importance
Extraeremos la importancia de cada variable para entender qué mira el modelo al predecir una cancelación.

In [ ]:
# Extraer los nombres de las columnas después del preprocesamiento
cat_encoder = rf_pipeline_final.named_steps['preprocessor'].named_transformers_['cat']
cat_features = cat_encoder.get_feature_names_out(vars_categoricas).tolist()

all_features = vars_numericas + cat_features + vars_binarias

# Extraer la importancia de las variables del Random Forest
importances = rf_pipeline_final.named_steps['classifier'].feature_importances_

# Crear un DataFrame para visualizar
df_importances = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Tomar el Top 15
top_15_features = df_importances.head(15)
print(top_15_features)

# Graficar
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=top_15_features, palette='viridis')
plt.title('Top 15 Variables más Importantes para Predecir Cancelaciones', fontsize=16, pad=20)
plt.xlabel('Importancia Relativa', fontsize=12)
plt.ylabel('Variable', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

El *Feature Importance* del Random Forest nos está contando una historia muy clara sobre el comportamiento de los clientes del hotel. 

### Interpretación de las Variables Más Importantes:

1. **`lead_time` (14.8%) - El factor rey:**
   * *Qué significa:* La cantidad de días entre la fecha en que se hizo la reserva y la fecha de llegada.
   * *Impacto de negocio:* Es lógico. Las reservas hechas con muchísima anticipación (ej. 6 meses antes) tienen una probabilidad altísima de ser canceladas porque los planes de la gente cambian. Las reservas de último minuto casi nunca se cancelan.

2. **`total_of_special_requests` (10.3%) - El factor de compromiso:**
   * *Qué significa:* Cuántas peticiones especiales hizo el cliente (ej. cama extra, piso alto).
   * *Impacto de negocio:* Los clientes que se toman el tiempo de hacer peticiones especiales están mucho más "comprometidos" con su estadía. Una reserva sin peticiones especiales es más propensa a ser una reserva "por si acaso" que terminará en cancelación.

3. **`is_portugal` (10.3%) - El factor local:**
   * *Qué significa:* Si el cliente es de Portugal (el país donde están los hoteles del dataset original) o extranjero.
   * *Impacto de negocio:* Los clientes locales suelen tener un comportamiento de cancelación muy distinto a los extranjeros. A menudo, las reservas locales son más volátiles (escapadas de fin de semana que se cancelan si llueve), mientras que los vuelos internacionales rara vez se cancelan.

4. **`agent` (7.7%) y `market_segment_Online TA` (5.7%) - El canal de venta:**
   * *Qué significa:* Quién hizo la reserva (qué agencia de viajes) y si vino por una Agencia de Viajes Online (OTA como Booking.com o Expedia).
   * *Impacto de negocio:* Las reservas hechas por Booking.com (Online TA) son famosas por su alta tasa de cancelación, ya que estas plataformas fomentan la "cancelación gratuita". El modelo ha detectado perfectamente este patrón del mercado.

5. **`adr` (7.4%) - El factor precio:**
   * *Qué significa:* La tarifa diaria promedio (Average Daily Rate).
   * *Impacto de negocio:* Las habitaciones más caras o las reservas hechas en temporada alta (cuando el ADR sube) tienen dinámicas de cancelación distintas.

Además, vemos otras variables muy interesantes en el Top 15:
* **`required_car_parking_spaces` (5.7%)**: Si alguien pide estacionamiento, casi seguro que va a ir (no cancela).
* **`market_segment_Online TA` (5.7%)**: Las reservas por Booking/Expedia cancelan mucho más.
* **`deposit_type_Non Refund` (2.5%)**: Obviamente, si pagaron un depósito no reembolsable, la tasa de cancelación se desploma.


### 3. Simulación de Impacto de Negocio

Para demostrar el valor del modelo, simularemos un escenario financiero. 
Asumiremos los siguientes costos ficticios (pero proporcionales a la realidad hotelera):
* **Ingreso por reserva exitosa (ADR promedio):** $100
* **Costo de Falso Negativo (No predecimos cancelación y la habitación queda vacía):** Pérdida de $100.
* **Costo de Falso Positivo (Predecimos cancelación pero el cliente sí viene):** Costo de $20 (por ofrecer un descuento innecesario para retenerlo o por sobrevender y tener que reubicarlo).
* **Beneficio de Verdadero Positivo (Predecimos cancelación y cancela):** Ahorramos $80 (porque pudimos revender la habitación a tiempo, menos un costo operativo de $20).

In [ ]:
# REEMPLAZA ESTE VALOR CON EL UMBRAL ÓPTIMO QUE OBTUVISTE EN EL NOTEBOOK 09
UMBRAL_OPTIMO = 0.3961  

# Para la simulación, usaremos las predicciones sobre el mismo set de entrenamiento
# (En un caso real estricto, esto se haría sobre un set de validación, pero para 
# propósitos de demostración de impacto de negocio, esto es suficiente).
y_proba = rf_pipeline_final.predict_proba(X_train_full)[:, 1]

# Generar predicciones con ambos umbrales
y_pred_default = (y_proba >= 0.50).astype(int)
y_pred_optimo = (y_proba >= UMBRAL_OPTIMO).astype(int)

def calcular_impacto_financiero(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Matriz de Costos/Beneficios
    costo_fn = fn * (-100)  # Pérdida total de la habitación
    costo_fp = fp * (-20)   # Costo de fricción/sobreventa
    ahorro_tp = tp * 80     # Recuperación por reventa
    
    impacto_total = costo_fn + costo_fp + ahorro_tp
    return impacto_total, cm

# Escenario 1: Sin Modelo (Asumimos que nadie cancela, todos son FN si cancelan)
y_pred_sin_modelo = np.zeros_like(y_train_full)
impacto_sin_modelo, _ = calcular_impacto_financiero(y_train_full, y_pred_sin_modelo)

# Escenario 2: Modelo con Umbral 0.50
impacto_default, cm_default = calcular_impacto_financiero(y_train_full, y_pred_default)

# Escenario 3: Modelo con Umbral Óptimo
impacto_optimo, cm_optimo = calcular_impacto_financiero(y_train_full, y_pred_optimo)

# Mostrar Resultados
print("="*50)
print("SIMULACIÓN DE IMPACTO FINANCIERO")
print("="*50)
print(f"1. Pérdida Sin Modelo:                 ${impacto_sin_modelo:,.2f}")
print(f"2. Balance con Modelo (Umbral 0.50):   ${impacto_default:,.2f}")
print(f"3. Balance con Modelo (Umbral {UMBRAL_OPTIMO}):   ${impacto_optimo:,.2f}")
print("-"*50)
ahorro_vs_nada = impacto_default - impacto_sin_modelo
mejora_vs_default = impacto_optimo - impacto_default
ahorro_optimo_vs_nada = impacto_optimo - impacto_sin_modelo
print(f"AHORRO DEFAULT VS NO USAR MODELO:        +${ahorro_vs_nada:,.2f}")
print(f"DINERO EXTRA POR AJUSTAR EL UMBRAL:    +${mejora_vs_default:,.2f}")
print("-"*50)
print(f"AHORRO TOTAL VS NO USAR MODELO:        +${ahorro_optimo_vs_nada:,.2f}")
print("="*50)

In [ ]:
# Visualización del Impacto Financiero
escenarios = ['Sin Modelo', 'Modelo (Umbral 0.50)', f'Modelo (Umbral {UMBRAL_OPTIMO})']
valores = [impacto_sin_modelo, impacto_default, impacto_optimo]
colores = ['#E15759', '#F28E2B', '#59A14F']

plt.figure(figsize=(10, 6))
bars = plt.bar(escenarios, valores, color=colores)

# Añadir los valores sobre las barras
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + (abs(yval)*0.05 if yval > 0 else -abs(yval)*0.1), 
             f'${yval:,.0f}', ha='center', va='bottom', fontweight='bold')

plt.title('Balance Financiero Simulado por Escenario', fontsize=16, pad=20)
plt.ylabel('Balance Neto ($)', fontsize=12)
plt.axhline(0, color='black', linewidth=1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()